# Decomposed Noisy FSim Ansatz

Build the Cirq chemistry ansatz, decompose every restricted `FSim` gate into `CZ + 1Q` gates, apply a location-aware empirical noise model, and measure the Hamiltonian from the noisy density matrix trace.

In [8]:
from __future__ import annotations

from pathlib import Path

import cirq
import numpy as np
from cirq.contrib.svg import SVGCircuit

from main_cursor_lib import (
    DEFAULT_AMP_DAMP_GAMMA,
    DEFAULT_DEPOL_PROB,
    DEFAULT_HIGH_CZ_MULTIPLIER,
    DEFAULT_LEAKAGE_APPROX_PROB,
    DEFAULT_PHASE_DAMP_GAMMA,
    LocationAwareDecomposedNoise,
    load_hamiltonian_paths,
    load_observable_h,
    ordered_parameter_symbols,
    params_per_layer,
    prepare_decomposed_ansatz_cirq,
    trace_energy,
)
from shot_measurement import (
    estimate_energy_from_noisy_rho_shots,
    run_mitigation,
)

# -------------------------------------------------------------------
# Configuration (3 layers = 21 parameters for H4 / num_spatial=4)
# -------------------------------------------------------------------
H_atom = 4
bond_length = 2.0
num_spatial = 4
ansatz_layers = 3

# Layer-3 baseline vector (same as docs/scripts/layer3_mitigation_sweep.py).
vqe_parameters = np.array([
    1.02328779,
    1.33825349,
    0.78981901,
    0.36255437,
    0.64652655,
    3.31926241,
    -0.07988963,
    0.22301099,
    0.4992053,
    1.3831351,
    5.50652141,
    2.7144864,
    2.14070825,
    -0.10898955,
    -0.2443687,
    -0.45224915,
    0.95313939,
    -1.18188991,
    -1.73092033,
    1.36303661,
    1.41789702,
])

# Finite-shot measurement options.
num_shots = 8192
measurement_scheme = "ogm"  # ogm|adaptive_shadows|derandomization|shadow_grouping_bernstein|shadow_grouping_inconfidence|direct_pauli
sampling_seed = 1234

# Readout calibration (top-level arguments).
p_0_success = np.array([0.93, 0.87, 0.87, 0.88, 0.90, 0.89, 0.87, 0.91])
p_1_success = np.array([0.95, 0.91, 0.95, 0.96, 0.94, 0.94, 0.92, 0.95])

apply_readout_noise = True
apply_rem = True

# Shadowgrouping integration config.
SHADOWGROUPING_ROOT = "/Users/zacharyhe/shadowgrouping"
ogm_file = f"{SHADOWGROUPING_ROOT}/haozhaowu/H4/hamil_class/ogm_outputs/OGM_H4_bond_{bond_length:.1f}.txt"

# Noise parameters (Weber/Sycamore-motivated defaults in main_cursor_lib).
random_seed = 1234
theta_std_dev = 0.026 * 2
phi_std_dev = 0.037 * 2
amp_damp_gamma = DEFAULT_AMP_DAMP_GAMMA
phase_damp_gamma = DEFAULT_PHASE_DAMP_GAMMA
depol_prob = DEFAULT_DEPOL_PROB
high_cz_multiplier = DEFAULT_HIGH_CZ_MULTIPLIER
leakage_approx_prob = DEFAULT_LEAKAGE_APPROX_PROB

# -------------------------------------------------------------------
# Mitigation mode (single switch).
# -------------------------------------------------------------------
# One of: "none" | "zne" | "cdr" | "both".
mitigation_mode = "both"

# ZNE sub-config (used by mode in {"zne", "both"}).
zne_scales = [1.0, 2.0, 3.0]
zne_fit_order = 1

# CDR sub-config (used by mode in {"cdr", "both"}).
cdr_num_circuits = 24
cdr_t_max = 32              # hard non-Clifford gate budget per training circuit
cdr_min_snap_fraction = 0.0  # optional diversity knob
cdr_seed = 7

print_circuit = True
workspace = Path.cwd()

ppl = params_per_layer(num_spatial)
expected_num_parameters = ansatz_layers * ppl
if len(vqe_parameters) != expected_num_parameters:
    raise ValueError(
        f"ansatz_layers={ansatz_layers} requires {expected_num_parameters} parameters "
        f"({ppl} per layer), got {len(vqe_parameters)}."
    )
if len(p_0_success) != 2 * num_spatial or len(p_1_success) != 2 * num_spatial:
    raise ValueError("Readout calibration arrays must have length 2 * num_spatial.")

print(f"Workspace: {workspace}")
print(f"Using {ansatz_layers} layers and {len(vqe_parameters)} parameters.")
print(f"Measurement scheme: {measurement_scheme} | shots={num_shots} | sampling_seed={sampling_seed}")
print(f"Mitigation mode: {mitigation_mode}")


Workspace: /Users/zacharyhe/cross_chips_sim
Using 3 layers and 21 parameters.
Measurement scheme: ogm | shots=8192 | sampling_seed=1234
Mitigation mode: both


In [9]:
# Build decomposed ansatz and Hamiltonian.
ansatz_circuit, ansatz_qubits = prepare_decomposed_ansatz_cirq(num_spatial, ansatz_layers)
observable_H = load_observable_h(workspace, ansatz_qubits, H_atom, bond_length)
_, hamiltonian_pkl_path, hamiltonian_text_path = load_hamiltonian_paths(workspace, H_atom, bond_length)
print(f"Hamiltonian candidates: {hamiltonian_pkl_path.name}, {hamiltonian_text_path.name}")

symbols = ordered_parameter_symbols(num_spatial, ansatz_layers)
resolver = cirq.ParamResolver(dict(zip(symbols, vqe_parameters)))
hamiltonian_matrix = observable_H.matrix(qubits=ansatz_qubits)
print(f"Number of parameters: {len(symbols)}")


Hamiltonian candidates: H4_bond_2.pkl, H4_bond_2_pauli_convention.txt
Number of parameters: 21


In [10]:
# Draw the decomposed circuit for visual inspection.
if print_circuit:
    SVGCircuit(ansatz_circuit)


In [11]:
# Apply location-aware noise to the decomposed circuit.
chem_noise_model = LocationAwareDecomposedNoise(
    amp_damp_gamma=amp_damp_gamma,
    phase_damp_gamma=phase_damp_gamma,
    depol_prob=depol_prob,
    high_cz_multiplier=high_cz_multiplier,
    leakage_approx_prob=leakage_approx_prob,
)
noisy_ansatz_circuit = ansatz_circuit.with_noise(chem_noise_model)
if print_circuit:
    print(f"Noisy circuit depth: {len(noisy_ansatz_circuit)} moments")


Noisy circuit depth: 220 moments


In [12]:
# Noisy density-matrix trace energy and shot-based energies.
noisy_simulator = cirq.DensityMatrixSimulator(seed=random_seed)
resolved_noisy_circuit = cirq.resolve_parameters(noisy_ansatz_circuit, resolver)
noisy_result = noisy_simulator.simulate(resolved_noisy_circuit, qubit_order=ansatz_qubits)
rho_noisy = noisy_result.final_density_matrix

trace_noisy_energy = trace_energy(hamiltonian_matrix, rho_noisy)
print(f"VQE parameters  Tr[H rho_noisy]: {trace_noisy_energy:.12f}")  # noisy trace

shot_est = estimate_energy_from_noisy_rho_shots(
    rho_noisy,
    observable_H,
    ansatz_qubits,
    num_shots=num_shots,
    measurement_scheme=measurement_scheme,
    p_0_success=p_0_success,
    p_1_success=p_1_success,
    apply_rem=apply_rem,
    apply_readout_noise=apply_readout_noise,
    sampling_seed=sampling_seed,
    ogm_file=ogm_file,
    shadowgrouping_root=SHADOWGROUPING_ROOT,
)
print(f"Shot estimate (unmitigated): {shot_est['energy_unmitigated']:.12f}")
print(f"Shot estimate (REM):         {shot_est['energy_rem']:.12f}")


VQE parameters  Tr[H rho_noisy]: -2.440902464632
Shot estimate (unmitigated): -2.340906826430
Shot estimate (REM):         -2.368397293788


In [13]:
# Exact noiseless energy for the fixed VQE parameters.
exact_simulator = cirq.Simulator(seed=random_seed)
resolved_exact_circuit = cirq.resolve_parameters(ansatz_circuit, resolver)
exact_state = exact_simulator.simulate(resolved_exact_circuit, qubit_order=ansatz_qubits).final_state_vector
exact_vqe_energy = np.vdot(exact_state, hamiltonian_matrix @ exact_state).real
print(f"VQE parameters  Exact noiseless energy: {exact_vqe_energy:.12f}")


VQE parameters  Exact noiseless energy: -2.971707567945


In [14]:
# CDR fitting (per-Pauli), aligned with the LiH notebook bottom cell.
target_resolver = {s: float(v) for s, v in zip(symbols, vqe_parameters)}

base_noise_cfg = {
    "amp_damp_gamma": amp_damp_gamma,
    "phase_damp_gamma": phase_damp_gamma,
    "depol_prob": depol_prob,
    "leakage_approx_prob": leakage_approx_prob,
    "high_cz_multiplier": high_cz_multiplier,
}

shot_cfg = {
    "num_shots": int(num_shots),
    "measurement_scheme": str(measurement_scheme),
    "apply_readout_noise": bool(apply_readout_noise),
    "sampling_seed": int(sampling_seed),
    "epsilon": 0.1,
    "ogm_file": ogm_file,
    "shadowgrouping_root": SHADOWGROUPING_ROOT,
}

readout_cal = {
    "p_0_success": p_0_success,
    "p_1_success": p_1_success,
}

cdr_cfg = {
    "num_circuits": int(cdr_num_circuits),
    "t_max": int(cdr_t_max),
    "min_snap_fraction": float(cdr_min_snap_fraction),
    "seed": int(cdr_seed),
    "cdr_fit_scope": "per_pauli",
}

if measurement_scheme == "ogm" and (
    (not Path(SHADOWGROUPING_ROOT).is_dir()) or (not Path(ogm_file).is_file())
):
    print("Skip CDR: need valid ogm_file and SHADOWGROUPING_ROOT (same as OGM shot cell).")
else:
    mit_cdr = run_mitigation(
        "cdr",
        ansatz_circuit=ansatz_circuit,
        observable_h=observable_H,
        qubits=ansatz_qubits,
        target_resolver=target_resolver,
        target_params=target_resolver,
        symbols=symbols,
        base_noise_cfg=base_noise_cfg,
        shot_cfg=shot_cfg,
        readout_cal=readout_cal,
        cdr_cfg=cdr_cfg,
        simulator_seed=int(random_seed),
    )

    print("CDR (per-pauli)")
    print(
        f"raw finite-shot (unmit / REM): {float(mit_cdr['unmit_target']):.12f} / {float(mit_cdr['rem_target']):.12f} Eh"
    )
    print(
        "cdr corrected (unmit / REM): "
        f"{float(mit_cdr['cdr_unmit_corrected']):.12f} / {float(mit_cdr['cdr_rem_corrected']):.12f} Eh"
    )
    print(f"reference exact noiseless: {exact_vqe_energy:.12f} Eh")

CDR (per-pauli)
raw finite-shot (unmit / REM): -2.361595992501 / -2.391455627618 Eh
cdr corrected (unmit / REM): -2.532201696350 / -2.535737440060 Eh
reference exact noiseless: -2.971707567945 Eh
